# Module 1: The Behaving Animal

**Spinning Up in Active Inference** | Notebook 1 of 16

---

Before we write a single equation, we need to understand what we are trying to explain. This module traces the intellectual lineage from animal behavior experiments to modern computational models. The core tension — **stimulus-response learning** vs. **internal world models** — maps directly onto the divide between model-free RL and Active Inference.

By the end of this notebook you will:
1. Know the key historical experiments that motivate computational models of behavior
2. Have a working neuro-nav environment and agent
3. See the first hints of the model-free vs. model-based distinction

**Prerequisites:** Python, NumPy, matplotlib basics.  
**Time:** ~45 minutes.

## 1.1 Thorndike's Puzzle Box (1898)

Edward Thorndike placed hungry cats inside wooden crates. The only way out — and to the food — was to pull a loop of string, press a lever, or step on a platform. He measured *escape latency*: how long it took the cat to get out.

The results were striking:
- **No sudden insight.** Escape times decreased gradually over trials.
- **No reasoning.** The cats appeared to stumble on the solution by accident, then repeat it.

Thorndike formulated the **Law of Effect**:

> *"Of several responses made to the same situation, those which are accompanied or closely followed by satisfaction to the animal will, other things being equal, be more firmly connected with the situation."*

This is, in essence, the first statement of **reinforcement learning**: actions followed by reward become more likely. It treats the animal as a black box — no internal representations needed.

## 1.2 Pavlov's Conditioning

Ivan Pavlov's dogs learned to salivate at the sound of a bell that predicted food. This **classical conditioning** demonstrated:

- **Stimulus-stimulus associations:** The bell (CS) becomes linked to food (US)
- **Prediction:** The animal learns to *anticipate* outcomes
- **Extinction:** Remove the food, and the response fades — but not completely (spontaneous recovery)

The key insight for computational neuroscience: learning is about **prediction**, not just about reward. This will become central when we discuss prediction errors in Module 3.

## 1.3 Skinner's Operant Conditioning

B.F. Skinner extended Thorndike's work with the *operant conditioning chamber* ("Skinner box"). He showed that **reinforcement schedules** — the pattern of reward delivery — powerfully shape behavior:

| Schedule | Description | Behavior Pattern |
|----------|-------------|------------------|
| Fixed Ratio (FR) | Reward every N responses | High rate, pause after reward |
| Variable Ratio (VR) | Reward on average every N responses | Highest, steadiest rate |
| Fixed Interval (FI) | Reward for first response after T seconds | Scalloped (accelerates near end) |
| Variable Interval (VI) | Reward for first response after avg T seconds | Low, steady rate |

Skinner's radical behaviorism claimed that **all** behavior could be explained by reinforcement history — no need to invoke thoughts, goals, or internal models. This philosophical stance maps directly onto **model-free RL**, which learns value functions from reward signals without building a model of the environment.

## 1.4 Tolman's Cognitive Maps (1948)

Edward Tolman challenged the behaviorist orthodoxy with a series of elegant rat experiments:

**Experiment 1: Latent Learning.** Three groups of rats ran a maze:
- Group 1: Rewarded from day 1 — learned quickly
- Group 2: Never rewarded — wandered aimlessly  
- Group 3: No reward for 10 days, then rewarded on day 11

Group 3 immediately performed as well as Group 1 on day 12. They had been learning the layout all along — they just had no *reason* to demonstrate it until the reward appeared.

**Experiment 2: Shortcut-taking.** Rats trained on a circuitous path would, when that path was blocked, choose a novel shortcut they had never taken — but which pointed toward the goal.

Tolman concluded that rats build **cognitive maps** — internal representations of the spatial structure of their environment — that are separate from any specific stimulus-response association.

> *"The brain is far more like a map control room than it is like an old-fashioned telephone exchange."*  
> — Tolman (1948)

This is the intellectual ancestor of **model-based RL** and **Active Inference**: the idea that agents build generative models of the world and use them to plan.

## 1.5 Ethology: Tinbergen's Four Questions

While experimental psychologists studied learning in boxes, ethologists like Niko Tinbergen and Konrad Lorenz studied animals in their natural habitats. Tinbergen proposed that any behavior can be analyzed at four levels:

1. **Causation (mechanism):** What neural/computational process produces the behavior?
2. **Development (ontogeny):** How does the behavior develop over the lifetime?
3. **Function (adaptation):** What is the behavior's survival value?
4. **Evolution (phylogeny):** How did the behavior evolve across species?

Most RL and AIF research focuses on questions 1 and 2: *what algorithm generates the behavior* and *how does it learn*. But Active Inference makes a distinctive claim about question 3: organisms that minimize free energy are the ones that survive. We will return to this in later modules.

## 1.6 The Key Tension: S-R vs. Internal Models

We can now see the central fault line in the study of behavior:

| | Behaviorism / S-R | Cognitive Maps / Internal Models |
|---|---|---|
| **What is learned** | Stimulus-response mappings | A model of the world |
| **How behavior is generated** | Look up the best response | Simulate possible futures, then act |
| **Key evidence** | Thorndike's gradual learning curves | Tolman's latent learning, shortcuts |
| **Computational descendent** | Model-free RL (Q-learning, SARSA) | Model-based RL, Active Inference |
| **What is optimized** | Cumulative reward | Surprise / Free energy |

This is not merely a historical debate. Modern neuroscience has found evidence for *both* systems in the brain, and much of contemporary research asks how they interact. Our curriculum traces both paths.

---

Now let us make these ideas concrete with code.

## 1.7 Hands-On: The neuro-nav Environment

We will use [neuro-nav](https://github.com/awjuliani/neuro-nav), a Python library designed for neuroscience-inspired RL experiments. It provides grid and graph environments that let us study navigation — exactly the setting Tolman used.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# neuro-nav imports
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate
from neuronav.agents.td_agents import TDQ

# Curriculum plotting utilities
import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve, plot_agent_trajectory,
    dual_perspective_box, COLORS
)

np.random.seed(42)
print("Imports successful!")

In [ ]:
# ── Create a Four Rooms environment ───────────────────────────────────────────
# This is a classic navigation benchmark: an 11x11 grid divided into four rooms
# connected by narrow doorways. The agent must find the goal.

env = GridEnv(GridSize.small, obs_type=GridObservation.index, template=GridTemplate.four_rooms)
obs = env.reset()

print(f"Grid size: {env.grid_size}x{env.grid_size}")
print(f"Number of states: {env.grid_size ** 2}")
print(f"Number of actions: 4 (up, down, left, right)")
print(f"Starting observation (state index): {obs}")
print(f"Goal position: {env.goal_pos}")

In [ ]:
# ── Visualize the environment ─────────────────────────────────────────────────
# Let's render the grid to see the four-rooms layout.

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
env.render(ax=ax)
ax.set_title("Four Rooms Environment\n(Tolman would have loved this)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── A random agent: Thorndike's cat on trial 1 ────────────────────────────────
# Before learning, the agent acts randomly — like a cat frantically
# trying different actions in the puzzle box.

def run_random_episode(env, max_steps=200):
    """Run one episode with a uniformly random policy."""
    obs = env.reset()
    positions = [tuple(env.agent_pos)]
    total_reward = 0

    for step in range(max_steps):
        action = np.random.randint(4)  # random action
        obs, reward, done, info = env.step(action)
        positions.append(tuple(env.agent_pos))
        total_reward += reward
        if done:
            break

    return positions, total_reward, step + 1

positions, reward, steps = run_random_episode(env)
print(f"Random agent: reached goal in {steps} steps (reward: {reward:.2f})")
print(f"Trajectory length: {len(positions)} positions")

In [ ]:
# ── Plot the random agent's trajectory ────────────────────────────────────────
# Notice the wandering, undirected path — no cognitive map here.

goal = tuple(env.goal_pos)
plot_agent_trajectory(
    positions, 
    grid_size=env.grid_size, 
    title="Random Agent Trajectory (Trial 1)\nLike Thorndike's cat: trial and error",
    goal_pos=goal
)
plt.tight_layout()
plt.show()

## 1.8 Learning from Reward: The TDQ Agent

Now let us give the agent the ability to learn from reward, like Thorndike's cats over repeated trials. The **TDQ** (Temporal-Difference Q-learning) agent learns a Q-value for each state-action pair:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

This is pure model-free learning: the agent stores cached values (S-R mappings) without building any model of the environment.

In [ ]:
# ── Train a TDQ agent for 100 episodes ────────────────────────────────────────

state_size = env.grid_size ** 2
action_size = 4

agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99, 
            poltype="softmax", beta=5.0)

n_episodes = 100
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs = env.reset()
    total_reward = 0
    done = False
    steps = 0

    while not done and steps < 500:
        action = agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        agent.update((obs, action, next_obs, reward, done))
        obs = next_obs
        total_reward += reward
        steps += 1

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

print(f"Training complete! Mean reward (last 10 eps): {np.mean(episode_rewards[-10:]):.3f}")
print(f"Mean episode length (last 10 eps): {np.mean(episode_lengths[-10:]):.1f} steps")

In [ ]:
# ── Plot the learning curve ───────────────────────────────────────────────────
# Like Thorndike's escape latency curves: gradual improvement over trials.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_learning_curve(
    {"TDQ Agent": episode_rewards}, 
    ylabel="Episode Reward",
    title="Learning Curve: Reward over Episodes",
    window=5,
    ax=ax1
)

plot_learning_curve(
    {"TDQ Agent": episode_lengths}, 
    ylabel="Steps to Goal",
    title="Learning Curve: Episode Length\n(cf. Thorndike\'s escape latency)",
    window=5,
    ax=ax2
)

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize Q-values as a heatmap: the agent's "cognitive map" ────────────
# The maximum Q-value at each state tells us how "close" the agent thinks
# it is to the goal. This is, in a sense, the agent's internal map —
# but it was learned from reward signals, not from a world model.

# Extract V(s) = max_a Q(s,a) for each state
q_table = agent.q_table  # shape: (state_size, action_size)
v_values = np.max(q_table, axis=1)  # shape: (state_size,)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_value_heatmap(
    v_values, env.grid_size,
    title="V(s) = max_a Q(s,a)\nThe agent's learned 'cognitive map'",
    cmap="inferno",
    goal_pos=goal,
    ax=ax1
)

# Show the trajectory of the trained agent
obs = env.reset()
trained_positions = [tuple(env.agent_pos)]
done = False
while not done and len(trained_positions) < 200:
    action = np.argmax(q_table[obs])  # greedy policy
    obs, reward, done, info = env.step(action)
    trained_positions.append(tuple(env.agent_pos))

plot_agent_trajectory(
    trained_positions, env.grid_size,
    title=f"Trained Agent Trajectory ({len(trained_positions)} steps)\nDirect path to goal",
    goal_pos=goal,
    ax=ax2
)

plt.tight_layout()
plt.show()

In [ ]:
# ── Dual Perspective: Behaviorism vs. Cognitive Maps ──────────────────────────

dual_perspective_box(
    rl_text=(
        "The TDQ agent learns Q(s,a) values — cached stimulus-response mappings. "
        "It never builds a model of the grid layout. It just knows: 'in state 42, "
        "action RIGHT gives me the highest expected reward.' This is Thorndike's "
        "Law of Effect implemented as an algorithm."
    ),
    aif_text=(
        "An Active Inference agent would build a generative model: transition matrices "
        "B(s'|s,a) encoding 'if I go right from state 42, I end up in state 43.' "
        "It could then plan novel routes to new goals without relearning — like Tolman's "
        "rats taking shortcuts. The value of an action comes from simulating its "
        "consequences, not from cached reward signals."
    ),
    title="Behaviorism (RL) vs. Cognitive Maps (Active Inference)"
)

## 1.9 Exercise: Exploring Different Environments

neuro-nav provides several grid templates that capture different experimental paradigms. Try each one and observe how the agent's exploration patterns differ.

Available templates:
- `GridTemplate.empty` — Open field, no obstacles
- `GridTemplate.four_rooms` — Classic four-rooms layout (used above)
- `GridTemplate.t_maze` — T-maze, a classic choice paradigm
- `GridTemplate.two_step` — Two-step task (tests model-based vs model-free)

**Questions to consider:**
1. How does the structure of the environment affect the shape of the value function?
2. In which environment does Q-learning struggle most? Why?
3. What would a model-based agent (Tolman's rat) do differently in the T-maze?

In [ ]:
# ── Exercise: Try different environments ──────────────────────────────────────

templates = [
    (GridTemplate.empty, "Empty Field"),
    (GridTemplate.four_rooms, "Four Rooms"),
    (GridTemplate.t_maze, "T-Maze"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (template, name) in enumerate(templates):
    env_t = GridEnv(GridSize.small, obs_type=GridObservation.index, template=template)

    # Train a fresh agent
    agent_t = TDQ(env_t.grid_size ** 2, 4, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0)
    for ep in range(100):
        obs = env_t.reset()
        done = False
        steps = 0
        while not done and steps < 500:
            action = agent_t.sample_action(obs)
            next_obs, reward, done, info = env_t.step(action)
            agent_t.update((obs, action, next_obs, reward, done))
            obs = next_obs
            steps += 1

    # Plot value function
    v = np.max(agent_t.q_table, axis=1)
    plot_value_heatmap(
        v, env_t.grid_size,
        title=f"{name}\nV(s) after 100 episodes",
        cmap="inferno",
        goal_pos=tuple(env_t.goal_pos),
        ax=axes[idx]
    )

plt.tight_layout()
plt.show()

## 1.10 Looking Ahead

In this module, we have seen that:

1. **Thorndike and Skinner** established that animals learn from reward — the intellectual foundation of RL
2. **Pavlov** showed that learning is fundamentally about **prediction** — a theme that will become central in Module 3 (dopamine and prediction errors)
3. **Tolman** demonstrated that animals build **internal models** of the world — the intellectual foundation of model-based RL and Active Inference
4. **Tinbergen** reminded us to ask *why* organisms behave as they do — Active Inference offers a principled answer

The Q-values we visualized above are a start, but they are limited: they are tied to a specific reward location and must be relearned from scratch if the goal changes. In Module 2, we will formalize the mathematics behind these values — the Bellman equation — and begin to see how Active Inference offers a different decomposition of the same problem.

**Next:** [Module 2: Reward, Value, and the Bellman Equation](02_bellman_and_value.ipynb)

---

## Further Reading

1. **Thorndike, E.L. (1898).** *Animal Intelligence: An Experimental Study of the Associative Processes in Animals.* The original puzzle-box experiments. Remarkably readable for a paper from 1898.

2. **Tolman, E.C. (1948).** *Cognitive Maps in Rats and Men.* Psychological Review, 55(4), 189-208. The classic paper that introduced cognitive maps. Essential reading for anyone interested in model-based reasoning.

3. **Dolan, R.J. & Dayan, P. (2013).** *Goals and Habits in the Brain.* Neuron, 80(2), 312-325. A modern neuroscience perspective on the model-free vs. model-based distinction, with evidence for both systems in the human brain.

4. **Sutton, R.S. & Barto, A.G. (2018).** *Reinforcement Learning: An Introduction* (2nd ed.), Chapter 1. The standard textbook. Chapters 1-2 cover the historical and conceptual foundations we discussed here.

5. **Friston, K.J. et al. (2017).** *Active Inference: A Process Theory.* Neural Computation, 29(1), 1-49. The foundational Active Inference paper. Dense, but Section 1 provides a good overview of how AIF relates to the behaviorist/cognitive tradition.